# Pipeline de transformation des données — Projet Spotify

Ce notebook transforme le dataset brut en **datasets dérivés** consommés par les
visualisations D3. Chaque section = une transformation qui lit un dataset et en
**produit un nouveau** (façon pipeline / notebook).

```
data/raw/spotify_songs.csv   (brut)
      │  00 nettoyage
      ▼
data/derived/tracks_clean.csv        (D0)
      │  01 normalisation        02 hiérarchie     03 agrégats     04 corrélation
      ▼                          ▼                 ▼               ▼
tracks_features.csv (D1)   hierarchy.json (D2)  trends.json (D3)  correlation.json (D4)
```

**Prérequis :** `pip install pandas numpy`
**Lancer :** ouvrir ce notebook depuis le dossier `scripts/` (Jupyter) et exécuter toutes les cellules.

In [1]:
import pandas as pd, numpy as np, json, os

ImportError: DLL load failed while importing lib: Une stratégie de contrôle d’application a bloqué ce fichier.

In [ ]:
# Chemins (relatifs au dossier scripts/)
RAW = "../data/raw/spotify_songs.csv"
OUT = "../data/derived"
os.makedirs(OUT, exist_ok=True)

# Features audio déjà dans [0,1]
FEATURES_01 = ["danceability","energy","valence","acousticness",
               "speechiness","instrumentalness","liveness"]
# Features à normaliser (échelles différentes)
FEATURES_NORM = ["tempo","loudness"]

## 00 · Nettoyage → `tracks_clean.csv` (D0)

- dédoublonnage sur `track_id` (un titre apparaît dans plusieurs playlists) ;
- `track_name` manquant → "Unknown Track" ;
- extraction de l'**année** depuis `track_album_release_date` (format `YYYY` ou `YYYY-MM-DD`) ;
- variables **calculées** : `decade`, `duration_min`, `mood = valence × energy`.

In [ ]:
df = pd.read_csv(RAW)
print("brut :", len(df), "lignes")

df = df.drop_duplicates(subset="track_id").copy()
df["track_name"] = df["track_name"].fillna("Unknown Track")
df["year"]     = df["track_album_release_date"].astype(str).str.extract(r"(\d{4})").astype(float)
df["decade"]   = (df["year"] // 10 * 10)
df["duration_min"] = df["duration_ms"] / 60000
df["mood"]     = df["valence"] * df["energy"]

clean_cols = (["track_id","track_name","track_artist","track_popularity",
               "playlist_genre","playlist_subgenre","year","decade","duration_min","mood"]
              + FEATURES_01 + FEATURES_NORM)
d0 = df[clean_cols].rename(columns={"playlist_genre":"genre",
                                    "playlist_subgenre":"subgenre",
                                    "track_popularity":"popularity",
                                    "track_artist":"artist"})
d0.to_csv(f"{OUT}/tracks_clean.csv", index=False)
print("D0 tracks_clean :", len(d0), "titres uniques")
d0.head(3)

## 01 · Normalisation → `tracks_features.csv` (D1)

Pour les **coordonnées parallèles** (multivarié), toutes les features doivent être
comparables : on normalise `tempo` et `loudness` en [0,1] (min-max). Les autres
features sont déjà dans [0,1].

In [ ]:
d1 = d0.copy()
for f in FEATURES_NORM:
    lo, hi = d1[f].min(), d1[f].max()
    d1[f + "_n"] = (d1[f] - lo) / (hi - lo)

feat_cols = (["track_id","track_name","artist","genre","subgenre","popularity"]
             + FEATURES_01 + ["tempo_n","loudness_n"])
d1[feat_cols].round(4).to_csv(f"{OUT}/tracks_features.csv", index=False)
print("D1 tracks_features :", len(d1), "titres x", len(FEATURES_01)+2, "features normalisées")

## 02 · Hiérarchie → `hierarchy.json` (D2)

Arbre `genre → sous-genre → artiste` (format `d3.hierarchy`). La valeur d'une
feuille = **nombre de titres** de l'artiste ; `dec` donne, **par décennie**,
`[nombre de titres, somme des popularités]` : la vue peut ainsi filtrer par décennie
et calculer la popularité moyenne sans relire les 28 000 titres. Alimente la **treemap / sunburst**.

In [ ]:
def build_hierarchy(data):
    # chaque feuille (artiste) garde, PAR DÉCENNIE : [nb de titres, somme des popularités]
    # -> la vue filtre par décennie et calcule la popularité moyenne sans relire les titres
    data = data.assign(artist=data["artist"].fillna("(artiste inconnu)"),
                       decade=data["decade"].astype(int))
    agg = (data.groupby(["genre", "subgenre", "artist", "decade"])["popularity"]
               .agg(n="size", ps="sum").reset_index())
    tree = {}
    for r in agg.itertuples(index=False):
        leaf = tree.setdefault(r.genre, {}).setdefault(r.subgenre, {}).setdefault(r.artist, {})
        leaf[str(r.decade)] = [int(r.n), int(r.ps)]
    return {"name": "Spotify", "children": [
        {"name": g, "children": [
            {"name": s, "children": [
                {"name": a, "value": sum(v[0] for v in dec.values()), "dec": dec}
                for a, dec in arts.items()]}
            for s, arts in subs.items()]}
        for g, subs in tree.items()]}

hier = build_hierarchy(d0)
with open(f"{OUT}/hierarchy.json", "w") as f:
    json.dump(hier, f, ensure_ascii=False, separators=(",", ":"))
print("D2 hierarchy :", len(hier["children"]), "genres")

## 03 · Agrégats temporels → `trends.json` (D3)

Moyenne de chaque feature + nombre de titres, par **décennie × genre**. Alimente
les **courbes multiples**.

> ⚠️ Le dataset est très concentré sur 2019 et pauvre avant 1985 → on agrège par
> décennie, et on le signalera dans les limites du rapport.

In [ ]:
agg_feats = FEATURES_01 + FEATURES_NORM + ["popularity"]
trends = (d0.dropna(subset=["decade"])
            .groupby(["decade","genre"])
            .agg(count=("track_id","size"), **{f:(f,"mean") for f in agg_feats})
            .reset_index())
trends["decade"] = trends["decade"].astype(int)
trends.round(4).to_json(f"{OUT}/trends.json", orient="records")
print("D3 trends :", len(trends), "lignes (décennie x genre)")

## 04 · Matrice de corrélation → `correlation.json` (D4)  ·  *heatmap*

Corrélation de **Pearson** entre toutes les features (+ popularité), calculée sur
tous les titres. C'est une variable **calculée** (non stockée). Alimente la
**heatmap de corrélation**.

In [ ]:
corr_feats = FEATURES_01 + FEATURES_NORM + ["popularity"]
corr = d0[corr_feats].corr(method="pearson").round(3)

d4 = {"features": corr_feats,
      "labels": {f: f.capitalize() for f in corr_feats},
      "matrix": corr.values.tolist()}
with open(f"{OUT}/correlation.json", "w") as f:
    json.dump(d4, f, ensure_ascii=False, indent=1)

print("D4 correlation :", len(corr_feats), "x", len(corr_feats))
print("  energy ~ loudness      =", corr.loc["energy","loudness"])
print("  energy ~ acousticness  =", corr.loc["energy","acousticness"])
corr

## 05 · Pays d'origine → `countries.json` (D5)  ·  *bonus (TODO)*

Le dataset **n'a pas** de colonne pays. L'enrichissement `artiste → pays d'origine`
se fera via **MusicBrainz** (variable calculée). À réaliser après le cœur du projet
— on prépare juste la liste des artistes à enrichir.

In [ ]:
artists = d0["artist"].dropna().unique()
d5 = {"status": "TODO",
      "method": "artiste -> pays d'origine via MusicBrainz (variable calculée)",
      "n_artists_a_enrichir": int(len(artists)),
      "by_country": []}
with open(f"{OUT}/countries.json", "w") as f:
    json.dump(d5, f, ensure_ascii=False, indent=1)
print("D5 countries : stub (", len(artists), "artistes à enrichir )")

## Récapitulatif

| Dataset dérivé | Vue qui le consomme |
|---|---|
| `tracks_features.csv` (D1) | Coordonnées parallèles |
| `hierarchy.json` (D2) | Treemap / sunburst |
| `trends.json` (D3) | Courbes multiples |
| `correlation.json` (D4) | Heatmap de corrélation |
| `countries.json` (D5) | Carte (bonus) |

Dernière transformation (côté D3) : le **mapping** de ces données vers les
**variables visuelles** de chaque technique.